# Uncertainty and hydraulic layers

A DEM is more useful when it states its own uncertainty and its own
hydraulic caveats. This notebook keeps the full record of how the
error bars were built — including the textbook route that failed its
validation and what that failure measured.

**You are here: 05.** What the DEM knows about its own errors, and the hydraulic caveats it declares.

```text
+- the evidence chain ------------------------------------------------+
|  datacube -> water masks -> per-pixel wet/dry series                |
|    01 what is estimable  ->  02 boundary audit  ->  03 operator vs  |
|    gauges  ->  04 elevations vs truth  ->  05 uncertainty and       |
|    hydraulic layers  ->  06 negatives kept  ->  07 coast census     |
+---------------------------------------------------------------------+
```

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from the repo root so the results/ paths resolve
here = Path.cwd()
while not (here / "pyintertidal").is_dir():
    if here.parent == here:
        raise FileNotFoundError("repo root not found above " + str(Path.cwd()))
    here = here.parent
os.chdir(here)

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## 1. What the transition width is made of
The fitted σ of each pixel's wet/dry transition mixes true sub-pixel
relief with the scene-level error of the assigned water level.
Because that level error enters the fit as scene-wide blur, it adds
in quadrature, and the terrain part is

$$\sigma_{\mathrm{topo}}=\sqrt{\max\big(\sigma^2-s_{\mathrm{level}}^2,\ \mathrm{floor}^2\big)},$$

with the floor at the fit grid's smallest atom — below it the
archive cannot distinguish relief from zero:

In [2]:
print("B2:", {k: round(v, 3) if isinstance(v, float) else v
              for k, v in load("results/b2_sigma/result.json").items()
              if k != "inputs_sha"})

B2: {'sd_level_m': 0.13, 'sigma_ajustada_mediana': 0.22, 'sigma_topo_mediana': 0.177, 'fraccion_varianza_nivel_mediana': 0.349, 'px_dominados_por_nivel': 11211, 'n_px': 33313}


### Concepts first: two kinds of error

Picture a kitchen scale. If its readings flutter a little between
weighings, averaging fixes it — that is **statistical** (random) error,
and more data beats it down as $1/\sqrt{n}$. If the scale is
*miscalibrated* and always reads 30 g heavy, averaging helps nothing —
that is **systematic** error, shared by every reading. A per-pixel error
formula can only see the first kind, because the second is common to all
pixels and invisible from inside any one fit. Keep this split in mind:
the section below is the measurement of exactly how much of our error is
of each kind — and the answer reorganised the project's priorities.

## 2. The analytic error bar, refuted by its own validation
The obvious per-pixel bar is the Cramér–Rao bound evaluated from each
pixel's own fit,

$$\sigma_z=\frac{s_e\,\sigma}{b\,\sqrt{\sum_t\varphi_t^2}},$$

with $s_e$ the pixel's residual noise, $b$ its wet-dry contrast and
$\varphi_t$ the Gaussian density at each observation's margin, so
$\sum_t\varphi_t^2$ counts how often the waterline was actually
caught crossing. If the bars are honest,
$u=(z-z_{\mathrm{ref}})/\sigma_z$ should be standard normal:
68 % of $|u|<1$ and 95 % of $|u|<1.96$. Against LiDAR truth it
fails completely — and the failure is the most useful number of the
week:

In [3]:
# if the bars told the truth: ~68%/95% coverage and Spearman > 0
r = load("results/p13_sigma_uncertainty/result.json")
print("formula:", r["formula"])
print(f"median predicted sigma_z: {r['sigma_z_mediana_m']:.3f} m "
      f"vs median |error|: {r['abs_error_mediana_m']:.3f} m")
print(f"coverage at 68/95% nominal: {100*r['cobertura_68']:.0f}% / "
      f"{100*r['cobertura_95']:.0f}%")
print(f"does it rank the errors? Spearman(|e|, sigma_z) = "
      f"{r['spearman_abs_e_vs_sigma_z']:+.3f}")

formula: sigma_z = s_e * sigma_p / (b_p * sqrt(sum phi^2))
median predicted sigma_z: 0.046 m vs median |error|: 0.309 m
coverage at 68/95% nominal: 9% / 17%
does it rank the errors? Spearman(|e|, sigma_z) = -0.006


The bound measures the statistical error of the fit — a few
centimetres — while the real error is an order of magnitude larger,
shared across pixels, and invisible to a per-pixel formula: it is the
assigned water level, not the fit. The fit is not the bottleneck.
That is the quantitative justification for the interior-tide work in
notebooks 01–03, and the reason the production σ_z is tabulated from
planted recoveries in the calibrated twin instead:

In [4]:
r = load("results/b7_incertidumbre/result.json")
print("sigma_z table (headroom x n_obs), metres:")
print(np.round(np.asarray(r["tabla_sigma_z"], float), 3))
print(f"dev coverage at 68% nominal: {100*r['cobertura_dev']:.0f}%")

sigma_z table (headroom x n_obs), metres:
[[  nan   nan 0.375]
 [  nan 0.158 0.673]
 [  nan 0.196 0.303]
 [  nan 0.144 0.169]
 [0.508 0.107 0.103]
 [  nan 0.126 0.107]
 [  nan   nan   nan]]
dev coverage at 68% nominal: 83%


### The calibrated twin, in plain words

You cannot compute the systematic error from inside the fit — but you can
*measure* it. Build a fake-but-realistic archive: invented terrain whose
elevations you know exactly, the same acquisition dates, the same real
cloud masks, noise calibrated on the real archive, and water levels
deliberately imperfect in the way the real ones are. Run the standard
estimator through it and record how wrong it comes out — there you *can*,
because you planted the answer. It is the method's flight simulator: you
examine the pilot in a storm you control. The error bars tabulated this
way pass the coverage exam against held-out truth (slightly conservative,
which is the right side to err on).

## 3. Ponding: spill elevation and depth, with both judges

In [5]:
r = load("results/b6_hydraulic_dem/result.json")
print(f"ponded pixels: {r['n_px_charco']:,} ({100*r['frac_charco']:.1f} %), "
      f"median depth {r['profundidad_mediana_m']:.2f} m")
print("real-signature judge:", r["juez_real"])
print("simulation judge (self-hiding censoring):", r["juez_simulacion"])

ponded pixels: 5,703 (14.3 %), median depth 1.22 m
real-signature judge: {'firma_observada': 0.04528213295785221, 'nulo_p95': 0.03884311372452991, 'nulo_media': 0.037533890414950755}
simulation judge (self-hiding censoring): {'precision': 0.6557377049180327, 'exhaustividad': 0.4024144869215292, 'n_plantados': 9463}
